In [5]:
!pip install selenium webdriver_manager pandas undetected-chromedriver

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47215 sha256=258a68e0552f5f3ff28186a8971d73890293aa6bdc3306d5db79552e96397da4
  Stored in directory: c:\users\ryan\appdata\local\pip\cache\wheels\5c\b9\03\4b6e38f019d6170e8c25df2e1e362d7bdf9ff4012df2dc85c0
Successfully built undetected-chromedriver



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import json
# BARIS BARU: Impor undetected_chromedriver sebagai uc
import undetected_chromedriver as uc
import pandas as pd
import time

# Opsi ini membantu menyamarkan automasi lebih jauh
options = uc.ChromeOptions()
options.add_argument('--start-maximized')
options.add_argument("--disable-blink-features=AutomationControlled")

# Ganti webdriver.Chrome menjadi uc.Chrome()
driver = uc.Chrome(options=options)

# Ganti dengan URL halaman yang Anda buka
url = 'https://www.olx.co.id/dijual-rumah-apartemen_c5158'

try:
    print("Membuka halaman dan menunggu data...")
    driver.get(url)
    
    # Beri waktu sedikit bagi halaman untuk "bernafas" sebelum mulai memeriksa
    time.sleep(5) 

    # Sekarang kita tunggu hingga datanya benar-benar ada
    print("Mencari objek window.__APP__...")
    # Kita tidak perlu lagi WebDriverWait karena undetected-chromedriver lebih canggih,
    # tapi kita pastikan datanya ada dengan loop sederhana
    app_data = None
    for _ in range(20): # Coba selama 20 detik
        app_data = driver.execute_script("return window.__APP__;")
        if app_data:
            print("Objek window.__APP__ ditemukan!")
            break
        time.sleep(1)
    
    if not app_data:
        raise Exception("Gagal menemukan window.__APP__ setelah menunggu.")

    # 2. Ambil semua detail iklan dari 'elements'
    all_elements = app_data['states']['items']['elements']
    # ... (sisa kode dari sini ke bawah tetap sama persis) ...
    collection_key = next(iter(app_data['states']['items']['collections']))
    item_ids = app_data['states']['items']['collections'][collection_key]

    print(f"Berhasil menemukan {len(item_ids)} ID iklan properti.")
    extracted_properties = []

    for item_id in item_ids:
        if item_id in all_elements:
            property_details = all_elements[item_id]
            title = property_details.get('title', 'N/A')
            price = property_details.get('price', {}).get('value', {}).get('raw', 0)
            location_resolved = property_details.get('locations_resolved', {})
            city = location_resolved.get('ADMIN_LEVEL_3_name', 'N/A')
            province = location_resolved.get('ADMIN_LEVEL_1_name', 'N/A')
            luas_bangunan = 'N/A'
            kamar_tidur = 'N/A'
            kamar_mandi = 'N/A'
            parameters = property_details.get('parameters', [])
            for param in parameters:
                if param.get('key') == 'p_sqr_building':
                    luas_bangunan = param.get('formatted_value')
                elif param.get('key') == 'p_bedroom':
                    kamar_tidur = param.get('formatted_value')
                elif param.get('key') == 'p_bathroom':
                    kamar_mandi = param.get('formatted_value')
            extracted_properties.append({
                'ID': item_id, 'Judul': title, 'Harga': price,
                'Kota': city, 'Provinsi': province,
                'Luas Bangunan (m2)': luas_bangunan,
                'Kamar Tidur': kamar_tidur, 'Kamar Mandi': kamar_mandi
            })

    df = pd.DataFrame(extracted_properties)
    print("\nData berhasil diekstrak:")
    print(df)

    df.to_csv('data_properti_olx.csv', index=False)
    print("\nData juga berhasil disimpan ke 'data_properti_olx.csv'")

except Exception as e:
    print(f"Terjadi kesalahan: {e}")

finally:
    # Tutup browser
    driver.quit()

Membuka halaman dan menunggu data...
Mencari objek window.__APP__...
Terjadi kesalahan: Gagal menemukan window.__APP__ setelah menunggu.
